Time Series Analysis: Tracking User Interactions Over Time

Objectives

1. Trend Analysis

      Understand daily, weekly, and monthly patterns for key user actions:

      Views (Product impression)

      Carts (Add-to-cart events)

      Purchases (Completed orders)

      Seasonality Detection

2. Identify recurring spikes or dips, e.g. during:

      Holidays

      Flash sales

      Weekends vs weekdays

      Conversion Timing

Measure average time lag between a product view and eventual purchase.

Data Preparation on Databricks (Sample Query)

In [0]:
%sql
-- Convert timestamps and bucket events by day
CREATE OR REPLACE TEMP VIEW daily_events AS
SELECT
  date(event_time) AS event_date,
  event_type,
  COUNT(*) AS event_count
FROM user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour
GROUP BY date(event_time), event_type;


In [0]:
import pandas as pd
import matplotlib.pyplot as plt

df = spark.sql("SELECT * FROM daily_events").toPandas()
pivot_df = df.pivot(index="event_date", columns="event_type", values="event_count").fillna(0)

pivot_df.plot(figsize=(14,6), title='Daily Events Trend (Views, Carts, Purchases)')
plt.xlabel("Date")
plt.ylabel("Event Count")
plt.grid(True)
plt.show()


Forecasting with Prophet for next 30 days

In [0]:
from prophet import Prophet

# Example: Forecasting Purchases
df_prophet = pivot_df.reset_index()[["event_date", "purchase"]].rename(columns={"event_date": "ds", "purchase": "y"})

model = Prophet()
model.fit(df_prophet)

future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

model.plot(forecast)
plt.title("Forecasted Purchases for Next 30 Days")
plt.show()


Anomaly Detection with Isolation Forest

In [0]:
from sklearn.ensemble import IsolationForest

# Detecting anomalies in views
data = pivot_df[["view"]].values
iso_forest = IsolationForest(contamination=0.05)
pivot_df["anomaly"] = iso_forest.fit_predict(data)

# Plot anomalies
plt.figure(figsize=(14,6))
plt.plot(pivot_df.index, pivot_df["view"], label="Views")
plt.scatter(pivot_df.index[pivot_df["anomaly"] == -1], 
            pivot_df["view"][pivot_df["anomaly"] == -1], 
            color='red', label='Anomaly')
plt.title("View Count with Anomalies")
plt.legend()
plt.show()


Average Time from View → Purchase

In [0]:
query = """
WITH views AS (
  SELECT user_id, MIN(event_time) AS view_time
  FROM user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour
  WHERE event_type = 'view'
  GROUP BY user_id
),
purchases AS (
  SELECT user_id, MIN(event_time) AS purchase_time
  FROM user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour
  WHERE event_type = 'purchase'
  GROUP BY user_id
)
SELECT
  AVG(DATEDIFF(p.purchase_time, v.view_time)) AS avg_days_to_purchase
FROM views v
JOIN purchases p ON v.user_id = p.user_id
WHERE p.purchase_time > v.view_time
"""
spark.sql(query).show()


Conversion Funnel Drop-off Analysis (View → Cart → Purchase)

Objective
Track how users transition from views → carts → purchases.

Identify conversion drop-off points by day or week.

Visualize trends to detect days or stages where users abandon.

Step 1: Prepare Daily Aggregates by Funnel Stage

In [0]:
%sql
-- Create a daily summary table by event type
CREATE OR REPLACE TEMP VIEW funnel_daily_events AS
SELECT
  date(event_time) AS event_date,
  event_type,
  COUNT(*) AS event_count
FROM user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour
WHERE event_type IN ('view', 'cart', 'purchase')
GROUP BY date(event_time), event_type;


In [0]:
%sql

select * from funnel_daily_events

Step 2: Pivot the Data into Funnel Columns

In [0]:
%sql
-- Pivot to get counts for each funnel stage
CREATE OR REPLACE TEMP VIEW funnel_pivot AS
SELECT
  event_date,
  COALESCE(MAX(CASE WHEN event_type = 'view' THEN event_count END), 0) AS views,
  COALESCE(MAX(CASE WHEN event_type = 'cart' THEN event_count END), 0) AS carts,
  COALESCE(MAX(CASE WHEN event_type = 'purchase' THEN event_count END), 0) AS purchases
FROM funnel_daily_events
GROUP BY event_date
ORDER BY event_date;


In [0]:
%sql

select * from funnel_pivot

Step 3: Add Drop-off Rate Calculation in PySpark

In [0]:
df = spark.sql("SELECT * FROM funnel_pivot").toPandas()

# Compute conversion rates
df["view_to_cart"] = (df["carts"] / df["views"]).round(3)
df["cart_to_purchase"] = (df["purchases"] / df["carts"]).round(3)
df["view_to_purchase"] = (df["purchases"] / df["views"]).round(3)

display(df)


Plot Funnel Drop-off Over Time

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
plt.plot(df["event_date"], df["view_to_cart"], label="View → Cart", marker='o')
plt.plot(df["event_date"], df["cart_to_purchase"], label="Cart → Purchase", marker='s')
plt.plot(df["event_date"], df["view_to_purchase"], label="View → Purchase", marker='^')
plt.title("Daily Funnel Conversion Rates Over Time")
plt.xlabel("Date")
plt.ylabel("Conversion Rate")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Flag High Drop-off Days

Dynamic Drop-off Thresholds

Percentile-Based Threshold (e.g., bottom 5%)

In [0]:
percentile_threshold = df["cart_to_purchase"].quantile(0.05)  # bottom 5%
high_dropoff_days = df[df["cart_to_purchase"] < percentile_threshold][["event_date", "cart_to_purchase"]]

print(f"🚨 Drop-off Threshold (5th percentile): {percentile_threshold:.3f}")
if not high_dropoff_days.empty:
    display(high_dropoff_days)
else:
    print("✅ No days found below the 5th percentile threshold.")


Standard Deviation-Based Threshold

In [0]:
mean_rate = df["cart_to_purchase"].mean()
std_dev = df["cart_to_purchase"].std()
std_dev_threshold = mean_rate - 2 * std_dev

high_dropoff_days_std = df[df["cart_to_purchase"] < std_dev_threshold][["event_date", "cart_to_purchase"]]

print(f"🚨 Drop-off Threshold (Mean - 2σ): {std_dev_threshold:.3f}")
if not high_dropoff_days_std.empty:
    display(high_dropoff_days_std)
else:
    print("✅ No days found below Mean - 2*Std Dev.")


######High Drop Off Days Analysis

Read Your Funnel Data

In [0]:
df = spark.sql("SELECT * FROM funnel_pivot").toPandas()

# Compute conversion rates
df["view_to_cart"] = (df["carts"] / df["views"]).round(3)
df["cart_to_purchase"] = (df["purchases"] / df["carts"]).round(3)
df["view_to_purchase"] = (df["purchases"] / df["views"]).round(3)


Compute Dynamic Threshold (Percentile-Based)

In [0]:
# Dynamic threshold using 5th percentile
percentile_threshold = df["cart_to_purchase"].quantile(0.05)

# Filter rows with low conversion
high_dropoff_days = df[df["cart_to_purchase"] < percentile_threshold][["event_date", "cart_to_purchase"]]


Display the Table in Notebook

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Convert back to Spark DataFrame if needed for `display()` in notebook
high_dropoff_spark_df = spark.createDataFrame(high_dropoff_days)
display(high_dropoff_spark_df)


Optional – Plot for Visual Clarity

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(df["event_date"], df["cart_to_purchase"], label="Cart → Purchase Rate", marker='o')
plt.axhline(y=percentile_threshold, color='red', linestyle='--', label=f"5th Percentile: {percentile_threshold:.2f}")
plt.title("Cart to Purchase Conversion Rate Over Time")
plt.xlabel("Date")
plt.ylabel("Conversion Rate")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [0]:
# Format date as 'YYYY-MM-DD'
high_dropoff_days["event_date"] = pd.to_datetime(high_dropoff_days["event_date"]).dt.strftime('%Y-%m-%d')

from IPython.display import display
display(high_dropoff_days)

